# Rozsa et al. (2023) on DANDI:000341, part 1: slice recordings without induction groups

Paper: Rozsa, Toth, Olah, Baka, Lakovics, Barzo and Tamas (2023), "Temporal disparity of action
potentials triggered in axon initial segments and distal axons in the neocortex", Sci. Adv. 9,
eade4511, doi:10.1126/sciadv.ade4511. Claims file: `claims/rozsa-2023.yaml`, written from the
version of record (Europe PMC full text) and its supplement.

Dataset: DANDI:000341, version 0.241220.1312 (doi:10.48324/dandi.000341/0.241220.1312). It is the
only published version; the paper cites the dandiset without a version, so the most recent
published version is pinned. The draft has the same 787 assets and size.

This notebook covers the ZD7288 recordings (folder `Persistent_RAF_ZD7288`), the soma and bleb
recordings (`Axonal_bleb_recording`), the active human slices (`In_vitro_Human_elevated_neuronal_activity`)
and the active rat slices with layer 5 LFP (`In_vitro_oscillation_Rat`). The induction recordings
are in `reproduce_rozsa_2023_induction.ipynb`. The in vivo recordings (folder `In_vivo`, 259
files, 383 GB) were not analysed; see the summary.

| Claim | Here |
|---|---|
| fig2c-zd7288-reduces-raf | tested |
| figs2-zd7288-passive-properties | tested |
| fig1hk-bleb-initiation-order | tested |
| fig2h-bleb-sag-higher | tested |
| fig3-human-ap-states | tested |
| fig3f-human-pre-ap-fluctuation | tested |
| fig3i-rat-lfp-rhythm | tested |
| fig3l-rat-spontaneous-ra-proportion | tested |
| fig3k-rat-pre-ap-fluctuation | tested |
| fig3m-rat-ais-phase-coupling | tested |
| fig3m-rat-ra-no-phase | tested |
| figs3-no-modulators | tested |
| the induction claims (fig1ab, fig1c, fig1d, fig1f, figs1b, figs1d, fig1g, fig2e, fig2il) | other notebook |
| the in vivo claims (fig4*, figs4i, figs5) | blocked, too_large |

In [ ]:
import zlib, time, resource, collections
import numpy as np
import pandas as pd
from scipy import stats, signal
import raf

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
PAPER = "rozsa-2023"


def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())


T0 = time.time()
assets = raf.list_assets()
groups = collections.defaultdict(list)
for a in assets:
    groups[a["path"].split("/")[0]].append(a)
print(len(assets), "assets in", raf.DANDISET, raf.VERSION)
for g, v in groups.items():
    print(f"{g:45s} files {len(v):4d}  subjects {len({a['path'].split('/')[1] for a in v}):3d}  "
          f"{sum(a['size'] for a in v) / 1e9:7.1f} GB")

Each file is read whole into memory (parallel range requests) and reduced at once to a table of
APs and a table of sweeps (`raf.extract`); the raw traces are not kept, except the LFP of the rat
recordings and the soma and bleb traces of the dual recordings.

In [ ]:
res = {}
for g, th in [("Persistent_RAF_ZD7288", 2), ("Axonal_bleb_recording", 2),
              ("In_vitro_Human_elevated_neuronal_activity", 2), ("In_vitro_oscillation_Rat", 1)]:
    t = time.time()
    keep = {"In_vitro_oscillation_Rat": ("LFP",), "Axonal_bleb_recording": ("Vm",)}.get(g, ())
    res[g] = raf.process_group(groups[g], keep=keep, threads=th)
    print(f"{g}: {len(res[g])} files in {time.time() - t:.0f} s; peak memory "
          f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")


def table(g):
    """One row per file: metadata, APs (main detection rule) and sweeps."""
    rows = []
    for p, (m, aps, sw, lfp) in res[g].items():
        rows.append(dict(path=p, file=p.split("/")[-1], species=m["subject_species"],
                         subject=m["subject_subject_id"], age=m.get("subject_age"),
                         meta=m, aps=aps, sw=sw, lfp=lfp))
    return rows

## Data check

In [ ]:
for g in res:
    rows = table(g)
    print("==", g)
    print("  files", len(rows), "species", dict(collections.Counter(r["species"] for r in rows)),
          "subjects", len({r["subject"] for r in rows}))
    sw = pd.concat([r["sw"] for r in rows])
    print("  channels", dict(collections.Counter(sw.chan)))
    print("  sampling rates (Hz)", dict(collections.Counter(sw.rate.round())))
    vm = sw[sw.chan.str.startswith("Vm")]
    print(f"  Vm read as volts x 1000: median {vm.vmed.median():.1f} mV, range of sweep minima "
          f"{vm.vmin.min():.1f} to {vm.vmax.max():.1f} mV")
    print("  sweeps whose stimulus is stored in pA though its unit is amperes:",
          int(vm["stim_in_pA"].sum()) if "stim_in_pA" in vm else 0,
          "; with values above 1e16 pA, stimulus ignored:", int(vm["stim_bad"].sum()) if "stim_bad" in vm else 0)
    print(f"  stimulus relative to holding: {np.nanmin(vm.imin):.0f} to {np.nanmax(vm.imax):.0f} pA; "
          f"holding {np.nanmin(vm.hold):.0f} to {np.nanmax(vm.hold):.0f} pA")
    aps = pd.concat([r["aps"] for r in rows if len(r["aps"])])
    print(f"  AP candidates {len(aps)}, main rule (peak > 0 mV) {int(aps.main.sum())}, "
          f"variant (peak >= threshold + 30 mV) {int(aps.variant30.sum())}")
    ph = collections.Counter()
    for r in rows:
        for k, v in r["meta"]["pharmacology"].items():
            if v != "NaN" and ("Drug" in k and "Name" in k or k.startswith("Drugname")):
                ph[v] += 1
    print("  drugs named in general/pharmacology:", dict(ph))
    # Recordings deposited twice: the same sweep count and the same median Vm of the first 20 sweeps.
    fp = collections.defaultdict(list)
    for r in rows:
        v = r["sw"][r["sw"].chan.str.startswith("Vm")].sort_values("start")
        fp[(len(v), tuple(v.vmed.round(3).head(20)))].append(r["file"])
    dup = [v for v in fp.values() if len(v) > 1]
    print("  files with identical sweep fingerprints:", dup if dup else "none")
print()
print("The voltage series give conversion 1e-12 with unit volts, but the stored values are in volts")
print("(medians near -65 V x 1e-3 above); read with the stated conversion they would be picovolts.")

Clock check for the rat LFP recordings. Firing and LFP deflections are said to be synchronized
(Figure 3I), so on a shared clock the AP count in 1 s bins should be correlated with the LFP at
lag 0, and less so when the LFP is shifted by several seconds. The raw Vm is not kept, so the AP
times stand in for the up states.

In [ ]:
def vm_lfp_corr(r, shifts=(0, 2, 5, 10)):
    """Correlation of the LFP (1 s means) with the AP count in the same 1 s bins, averaged over
    LFP segments of at least 20 s with 5 APs or more, at LFP shifts in seconds."""
    if not r["lfp"] or not len(r["aps"]):
        return None
    out = {}
    t_ap = r["aps"][r["aps"].main].t_abs.to_numpy()
    for sh in shifts:
        cs = []
        for (st, rate, x) in r["lfp"].values():
            dur = len(x) / rate
            if dur < 20:
                continue
            nb = int(dur)
            lf = x[: nb * int(rate)].reshape(nb, int(rate)).mean(1)
            cnt = np.histogram(t_ap, bins=st + sh + np.arange(nb + 1))[0]
            if cnt.sum() >= 5 and np.std(lf) > 0:
                cs.append(np.corrcoef(lf, cnt)[0, 1])
        out[sh] = np.nanmean(cs) if cs else np.nan
    return out


rows_osc = table("In_vitro_oscillation_Rat")
cc = pd.DataFrame({r["file"]: vm_lfp_corr(r) for r in rows_osc}).T
print("Mean correlation of the 1 s LFP mean with the AP count, by LFP shift (s):")
print(cc.describe().round(3))

## fig2c-zd7288-reduces-raf

> Blocking HCN channels with ZD7288 (30 uM) reduces the duration of RAF (47.2 to 10.1 s) and the
> number of RA-APs (624 to 48) in human and rat layer 1 interneurons.

The wash-in time of ZD7288 is in each file's `general/pharmacology` (`DrugWashinTime1`, in
seconds on the file clock, checked below against the sweep times). Control is the part of the
recording before it, ZD7288 the part after. In each part, RA-APs by reading (c) of the claims
file's header (`raf.ra_aps_induction`): APs without positive current after the first induction
sweep of that part. RAF duration and RA-AP count as in fig1ab-raf-patterns, reading (i), one
epoch per part. The Methods count the RA-APs of the continuous recording ("Analysis of RAF
patterns": "number of RA-APs during continuous recording"); the APs in the interstimulus
intervals of the induction are counted in the main reading and left out in a variant.

In [ ]:
def split_raf(aps, sw, t_from, t_to, continuous_only=False):
    a = aps[(aps.t_abs >= t_from) & (aps.t_abs < t_to)]
    s = sw[(sw.start >= t_from) & (sw.start < t_to)]
    ais, ra, t0, runs = raf.ra_aps_induction(a, s)
    if continuous_only and len(ra):
        stepless = set(s[s.nsteps == 0].series)
        ra = ra[ra.series.isin(stepless)]
    ep = raf.epochs(ra.t_abs)
    return dict(n_ra=len(ra), duration=ep[0]["duration"] if ep else 0.0,
                n_ind=len(raf.induction_sweeps(s)), t_ind=t0, ais=len(ais))


zd_rows = []
for r in table("Persistent_RAF_ZD7288"):
    ph = r["meta"]["pharmacology"]
    T = float(ph["DrugWashinTime1"])
    aps = r["aps"][r["aps"].main]
    sw = r["sw"]
    end = (sw.start + sw.n / sw.rate).max()
    for cont in (False, True):
        c = split_raf(aps, sw, -np.inf, T, cont)
        z = split_raf(aps, sw, T, np.inf, cont)
        zd_rows.append(dict(file=r["file"], species=r["species"].split()[0], drug=ph["DrugName1"],
                            conc=ph["DrugConcentration1"], washin_s=T, rec_end_s=round(end, 1),
                            continuous_only=cont,
                            ctrl_ind_sweeps=c["n_ind"], zd_ind_sweeps=z["n_ind"],
                            ctrl_dur=c["duration"], zd_dur=z["duration"],
                            ctrl_n=c["n_ra"], zd_n=z["n_ra"]))
zd = pd.DataFrame(zd_rows)
print(zd.round(2).to_string())

In [ ]:
for cont, d in zd.groupby("continuous_only"):
    print(f"== RA-APs {'of the continuous recording only' if cont else 'after induction, ISIs included'}"
          f" (n = {len(d)} cells)")
    for q in ("dur", "n"):
        c, z = d[f"ctrl_{q}"], d[f"zd_{q}"]
        p_t = stats.ttest_rel(c, z).pvalue
        rule = raf.paper_test(c, z, paired=True)
        print(f"  {q:4s} control mean {c.mean():8.1f} median {c.median():8.1f} | ZD7288 mean "
              f"{z.mean():8.1f} median {z.median():8.1f} | paired t P = {p_t:.4f}; "
              f"paper's rule: {rule[0]} P = {rule[1]:.4f}; cells decreasing {int((z < c).sum())}/{len(d)}")
    for sp, ds in d.groupby("species"):
        print(f"    {sp}: duration control {ds.ctrl_dur.round(1).tolist()} ZD {ds.zd_dur.round(1).tolist()}; "
              f"count control {ds.ctrl_n.tolist()} ZD {ds.zd_n.tolist()}")
print("expected: duration 47.2 -> 10.1 s, count 624 -> 48 (tolerance 10%), paired t P = 0.008 and 0.007")

**Result: not reproduced.** Across the 7 cells RAF duration does not fall after the ZD7288 wash-in time (mean 131.6 to 360.5 s with the interstimulus intervals, 85.1 to 195.5 s on the continuous recording) and the RA-AP count falls in 5 of 7 cells without significance (paired t P = 0.83 and 0.80). The three rat cells lose nearly all RA-APs (363, 919, 181 to 16, 14, 20).

## figs2-zd7288-passive-properties

> ZD7288 increases input resistance (70 to 98 MOhm, P = 0.01), and hyperpolarizes the resting
> potential (-60 to -67 mV, P = 0.075) and decreases sag fraction (0.04 to 0.017, P = 0.057)
> without reaching significance.

From the sweeps with a single hyperpolarizing step (`raf._sag`), before and after the wash-in
time. Input resistance: the steady deflection over the current of the smallest hyperpolarizing
step (criterion; standard choice); resting potential: the median of the pre-step baselines;
sag fraction as in fig2e-sag-species under reading (a) on the most hyperpolarizing step, with
reading (b) and the step whose peak is nearest -90 mV shown as variants. The pre-step baseline
includes any holding current; cells with a holding current in a period are listed.

In [ ]:
def passive(sw):
    s = sw.dropna(subset=["sag_a"]) if "sag_a" in sw else sw.iloc[:0]
    if not len(s):
        return dict(n=0)
    small = s[s.i_step == s.i_step.max()]
    big = s[s.i_step == s.i_step.min()]
    near = s.iloc[[int(np.argmin(np.abs(s.vpeak.to_numpy() + 90)))]]
    return dict(n=len(s), r_in=small.r_in.median(), v_rest=s.base.median(),
                hold=float(np.nanmedian(s.hold)),
                sag_a_big=big.sag_a.median(), sag_b_big=big.sag_b.median(),
                sag_a_90=near.sag_a.iloc[0], sag_b_90=near.sag_b.iloc[0],
                step_small=small.i_step.iloc[0], step_big=big.i_step.iloc[0])


pp = []
for r in table("Persistent_RAF_ZD7288"):
    T = float(r["meta"]["pharmacology"]["DrugWashinTime1"])
    sw = r["sw"][r["sw"].chan == "Vm"]
    for per, s in (("control", sw[sw.start < T]), ("ZD7288", sw[sw.start >= T])):
        pp.append(dict(file=r["file"], period=per, **passive(s)))
pp = pd.DataFrame(pp)
print(pp.round(3).to_string())

In [ ]:
w = pp.pivot(index="file", columns="period")
ok = w["n"].min(axis=1) > 0
print(f"cells with step sweeps in both periods: {int(ok.sum())} of {len(w)}")
w = w[ok]
for q, exp in (("r_in", (70, 98)), ("v_rest", (-60, -67)), ("sag_a_big", (0.04, 0.017)),
               ("sag_b_big", None), ("sag_a_90", None), ("sag_b_90", None)):
    c, z = w[q]["control"].astype(float), w[q]["ZD7288"].astype(float)
    m = ~(c.isna() | z.isna())
    c, z = c[m], z[m]
    p = stats.ttest_rel(c, z).pvalue if len(c) > 1 else np.nan
    print(f"{q:10s} n={len(c)} control median {c.median():8.3f} mean {c.mean():8.3f} | ZD7288 median "
          f"{z.median():8.3f} mean {z.mean():8.3f} | paired t P = {p:.3f}"
          + (f" | expected {exp[0]} -> {exp[1]}" if exp else ""))

**Result: not reproduced.** Input resistance does not rise (median 146.2 to 141.0 MOhm, paired t P = 0.78); resting potential (-63.6 to -63.4 mV) and sag fraction (0.056 to 0.047) do not change. Two of six values, the resting potentials, are within tolerance.

## fig1hk-bleb-initiation-order

> In dual soma and axonal bleb recordings of human NGFCs, AIS-APs reach the soma first and the
> bleb second, while RA-APs reach the bleb first, and RA-APs have lower somatic threshold and
> pre-AP slope.

The cell class is in each file's `general/pharmacology` (`celltype`: `NGF` or `no recovery`).
Somatic and bleb sweeps are paired by sweep number (`Vm_soma_<k>`, `Vm_bleb_<k>`). For each
somatic AP, the bleb AP within 3 ms is taken (a standard choice); latency is bleb time minus
soma time, at the threshold and at the peak (the criterion's two readings). AIS-APs and RA-APs by
reading (b), a Gaussian mixture on threshold and slope fitted to the somatic APs of the human
bleb recordings pooled; the dandiset has no AP labels, so reading (a) cannot be run.

In [ ]:
def bleb_latency(r, win_ms=3.0):
    """Soma-to-bleb latency for each somatic AP of a dual recording. The bleb signal is small (its
    peaks are mostly below 0 mV), so bleb APs are not detected on their own: in the bleb sweep of the
    same number, the bleb peak is the maximum within 3 ms of the somatic peak and the bleb threshold
    the first sample of the run with dV/dt >= 5 mV/ms before the steepest point ahead of that peak,
    as for somatic APs (standard choices). Latencies with no bleb run reaching 5 mV/ms are NaN."""
    aps = r["aps"]
    if not len(aps):
        return pd.DataFrame()
    so = aps[aps.main & (aps.chan == "Vm_soma")].copy()
    raw = r["lfp"]
    out = []
    for _, a in so.iterrows():
        k = a.series.rsplit("_", 1)[1]
        b = raw.get(f"Vm_bleb_{k}")
        row = a.to_dict()
        if b is not None:
            st, rate, x = b
            ms = rate / 1000
            lo, hi = int((a.t_peak - win_ms / 1000) * rate), int((a.t_peak + win_ms / 1000) * rate)
            lo, hi = max(lo, 1), min(hi, len(x) - 1)
            if hi - lo > 4:
                seg = x[lo:hi].astype(float)
                pk = lo + int(np.argmax(seg))
                dv = np.diff(x[lo - 1:pk + 1].astype(float)) * ms
                row["bleb_amp"] = float(x[pk] - np.median(x[max(0, lo - int(5 * ms)):lo]))
                row["lat_peak_ms"] = (pk / rate - a.t_peak) * 1000
                if len(dv) and dv.max() >= raf.DVDT:
                    j = int(np.argmax(dv))
                    while j > 0 and dv[j - 1] >= raf.DVDT:
                        j -= 1
                    row["lat_thr_ms"] = ((lo - 1 + j) / rate - a.t_thr) * 1000
                else:
                    row["lat_thr_ms"] = np.nan
        out.append(row)
    return pd.DataFrame(out)


bleb = []
for r in table("Axonal_bleb_recording"):
    ph = r["meta"]["pharmacology"]
    d = bleb_latency(r)
    n_bleb_sweeps = int((r["sw"].chan == "Vm_bleb").sum())
    bleb.append(dict(file=r["file"], species=r["species"].split()[0], celltype=ph.get("celltype"),
                     distance_um=float(ph["Electrode Distance (µm) xyz"]) if "Electrode Distance (µm) xyz" in ph else np.nan,
                     comment=ph.get("Comment", ""), bleb_sweeps=n_bleb_sweeps, pairs=d))
print(pd.DataFrame([{k: v for k, v in b.items() if k != "pairs"} | {
    "soma_APs": len(b["pairs"]), "with_bleb_sweep": int(b["pairs"].lat_peak_ms.notna().sum()) if "lat_peak_ms" in b["pairs"] else 0,
    "bleb_thr_found": int(b["pairs"].lat_thr_ms.notna().sum()) if "lat_thr_ms" in b["pairs"] else 0}
    for b in bleb]).to_string())

In [ ]:
hum = [b for b in bleb if b["species"] == "Homo" and len(b["pairs"])]
allh = pd.concat([b["pairs"].assign(file=b["file"], celltype=b["celltype"]) for b in hum])
allh["ra"] = raf.gmm_classes(allh, seed("fig1hk-bleb-initiation-order"))
print(f"human somatic APs {len(allh)}; RA-APs by the mixture {int(allh.ra.sum())}")
for (f, ct), d in allh.groupby(["file", "celltype"]):
    for cls, dd in d.groupby("ra"):
        name = "RA-AP " if cls else "AIS-AP"
        print(f"{f[:32]} {ct:12s} {name} n={len(dd):4d} with bleb sweep {int(dd.lat_peak_ms.notna().sum()) if 'lat_peak_ms' in dd else 0:4d} "
              f"median latency thr {dd.get('lat_thr_ms', pd.Series(dtype=float)).median():6.3f} ms "
              f"peak {dd.get('lat_peak_ms', pd.Series(dtype=float)).median():6.3f} ms | "
              f"threshold {dd.thr.median():6.1f} mV slope {dd.slope.median():5.2f} mV/ms")
# Rat recordings, for comparison (not judged)
rat = [b for b in bleb if b["species"] == "Rattus" and len(b["pairs"])]
allr = pd.concat([b["pairs"].assign(file=b["file"], celltype=b["celltype"]) for b in rat])
allr["ra"] = raf.gmm_classes(allr, seed("fig1hk-bleb-initiation-order") + 1)
print("rat, for comparison:")
for c in ("lat_thr_ms", "lat_peak_ms", "bleb_amp"):
    if c not in allr:
        allr[c] = np.nan
print(allr.groupby(["file", "celltype", "ra"]).agg(n=("thr", "size"), lat_thr=("lat_thr_ms", "median"),
                                       lat_peak=("lat_peak_ms", "median"), bleb_amp=("bleb_amp", "median"),
                                       thr=("thr", "median"), slope=("slope", "median")).round(3))

**Result: not reproduced.** In the NGFC 220504 the bleb leads for AIS-APs (median latency -0.22 ms at threshold) and RA-APs alike (-0.17 ms); in 220505-2 all 62 somatic APs fall in the AIS class (+0.16 ms) and none in the RA class.

## fig2h-bleb-sag-higher

> Axonal blebs show about four times the sag fraction of the soma of the same layer 1
> interneurons (0.4 against 0.1), at 106 um Euclidean soma-bleb distance.

Each compartment's sag from the hyperpolarizing steps injected into it (soma trace with a
somatic step, bleb trace with a bleb step), per cell, under the claim's readings: formula (a) or
(b), on the most hyperpolarizing step or the step whose peak is nearest -90 mV. A two-sample t
test across cells as the paper names it; a paired t test as a variant. The variant with the bleb
trace during a somatic step was not run (see the summary). The distance is the file's
`Electrode Distance (µm) xyz`.

In [ ]:
def sag_pick(s, how):
    s = s.dropna(subset=["sag_a"]) if "sag_a" in s else s.iloc[:0]
    if not len(s):
        return None
    if how == "most":
        return s[s.i_step == s.i_step.min()].iloc[0]
    return s.iloc[int(np.argmin(np.abs(s.vpeak.to_numpy() + 90)))]


rows = []
for r in table("Axonal_bleb_recording"):
    for how in ("most", "near90"):
        so = sag_pick(r["sw"][r["sw"].chan == "Vm_soma"], how)
        bl = sag_pick(r["sw"][r["sw"].chan == "Vm_bleb"], how)
        rows.append(dict(file=r["file"][:30], species=r["species"].split()[0], step=how,
                         soma_a=so.sag_a if so is not None else np.nan, bleb_a=bl.sag_a if bl is not None else np.nan,
                         soma_b=so.sag_b if so is not None else np.nan, bleb_b=bl.sag_b if bl is not None else np.nan,
                         soma_step=so.i_step if so is not None else np.nan, bleb_step=bl.i_step if bl is not None else np.nan))
sg = pd.DataFrame(rows)
print(sg.round(3).to_string())
for (how), d in sg.groupby("step"):
    for f in ("a", "b"):
        s_, b_ = d[f"soma_{f}"].dropna(), d[f"bleb_{f}"].dropna()
        both = d.dropna(subset=[f"soma_{f}", f"bleb_{f}"])
        p2 = stats.ttest_ind(b_, s_).pvalue
        pp_ = stats.ttest_rel(both[f"bleb_{f}"], both[f"soma_{f}"]).pvalue if len(both) > 1 else np.nan
        print(f"step {how:6s} formula ({f}): soma n={len(s_)} median {s_.median():.3f} mean {s_.mean():.3f} | "
              f"bleb n={len(b_)} median {b_.median():.3f} mean {b_.mean():.3f} | two-sample t P = {p2:.2g}; "
              f"paired t P = {pp_:.2g} (n={len(both)})")
dist = pd.Series([b["distance_um"] for b in bleb])
print(f"soma-bleb distance: n={dist.notna().sum()} median {dist.median():.0f} um mean {dist.mean():.0f} um "
      f"range {dist.min():.0f} to {dist.max():.0f}")
print("expected: soma 0.1, bleb 0.4 (tolerance 10%), two-sample t P < 0.001, distance 106 um")

**Result: partially reproduced.** The bleb sag is higher than the somatic one (0.115 against 0.024, two-sample t P = 0.0082, most hyperpolarizing step, formula (a)), but not at P < 0.001, and neither value is within tolerance; the step nearest -90 mV shows no difference (P = 0.15). The median soma-bleb distance is 106 um.

## fig3-human-ap-states

> In active human slices (n = 26 cells), AIS-APs occur only in depolarized up states in all 7
> cells with AIS-APs, while spontaneous RA-APs, seen in 6 cells, also occur in hyperpolarized
> down and transition states.

APs of the sweeps without current injection only (the claim's "without current injection").
AIS-APs and RA-APs by reading (b): a Gaussian mixture on threshold and slope fitted to these APs
of the 26 human active-slice recordings pooled. The dandiset carries no AP labels (reading (a));
five files have a `Time To first RAAP (s)` entry in `general/pharmacology`, shown beside the
result. States per cell from the histogram of the membrane potential of the sweeps without
current, by the criterion's readings: (i) a two-Gaussian fit, down and up within one SD of each
mean, transition between; (ii) the range between the two means cut in thirds. The AP's
potential is the mean 0 to 50 ms before threshold, or the threshold.

In [ ]:
def no_current_aps(r):
    a, s = r["aps"], r["sw"]
    if not len(a):
        return a
    quiet = set(s[(s.nsteps == 0) & s.chan.str.startswith("Vm")].series)
    return a[a.main & a.series.isin(quiet)]


def states(hist, sd):
    """Two-Gaussian fit to a membrane potential histogram (bins raf.HIST_BINS); returns the bounds
    of down and up states under readings (i) and (ii)."""
    from sklearn.mixture import GaussianMixture
    centers = (raf.HIST_BINS[:-1] + raf.HIST_BINS[1:]) / 2
    w = np.round(hist / hist.sum() * 20000).astype(int)
    x = np.repeat(centers, w)[:, None]
    gm = GaussianMixture(2, random_state=sd).fit(x)
    order = np.argsort(gm.means_[:, 0])
    mu = gm.means_[order, 0]
    sdv = np.sqrt(gm.covariances_[order, 0, 0])
    i_down, i_up = mu[0] + sdv[0], mu[1] - sdv[1]
    if i_down > i_up:
        i_down = i_up = (mu[0] + mu[1]) / 2
    third = (mu[1] - mu[0]) / 3
    return dict(mu_down=mu[0], mu_up=mu[1], sd_down=sdv[0], sd_up=sdv[1],
                i_down=i_down, i_up=i_up, ii_down=mu[0] + third, ii_up=mu[1] - third)


def classify_state(v, b, reading):
    lo, hi = (b["i_down"], b["i_up"]) if reading == "i" else (b["ii_down"], b["ii_up"])
    return np.where(v <= lo, "down", np.where(v >= hi, "up", "transition"))


hrows = table("In_vitro_Human_elevated_neuronal_activity")
H = pd.concat([no_current_aps(r).assign(file=r["file"]) for r in hrows if len(no_current_aps(r))])
H["ra"] = raf.gmm_classes(H, seed("fig3-human-ap-states"))
print(f"human active slices: {len(hrows)} files, {len(H)} APs without current, RA-APs {int(H.ra.sum())}")
print(H.groupby("ra")[["thr", "slope"]].describe().round(2).T)
bounds = {}
for r in hrows:
    s = r["sw"]
    hs = [h for h in s.get("hist", pd.Series(dtype=object)).dropna()] if "hist" in s else []
    if hs:
        bounds[r["file"]] = states(np.sum(hs, axis=0), seed("fig3-human-ap-states"))
print(pd.DataFrame(bounds).T.round(1).to_string())

In [ ]:
tf = {r["file"]: r["meta"]["pharmacology"].get("Time To first RAAP (s)", "NaN") for r in hrows}
res_states = {}
for reading in ("i", "ii"):
    for pot in ("vpre", "thr"):
        rows = []
        for f, d in H.groupby("file"):
            if f not in bounds:
                continue
            st = classify_state(d[pot].to_numpy(), bounds[f], reading)
            ais, ra = st[~d.ra.to_numpy()], st[d.ra.to_numpy()]
            rows.append(dict(file=f[:30], n_ais=len(ais), ais_up=int((ais == "up").sum()),
                             n_ra=len(ra), ra_down=int((ra == "down").sum()), ra_trans=int((ra == "transition").sum()),
                             ra_up=int((ra == "up").sum()), label_time_to_first_RAAP=tf[f]))
        t = pd.DataFrame(rows)
        res_states[(reading, pot)] = t
        n_ais = int((t.n_ais > 0).sum())
        all_up = int(((t.n_ais > 0) & (t.ais_up == t.n_ais)).sum())
        n_ra = int((t.n_ra > 0).sum())
        ra_dt = int(((t.n_ra > 0) & ((t.ra_down + t.ra_trans) > 0)).sum())
        print(f"reading ({reading}), potential {pot}: cells with AIS-APs {n_ais} (expected 7), all AIS-APs "
              f"in up states in {all_up}; cells with RA-APs {n_ra} (expected 6), with RA-APs in down or "
              f"transition states {ra_dt}")
print(res_states[("i", "vpre")].to_string())
print("files with a 'Time To first RAAP' entry:", {k[:30]: v for k, v in tf.items() if v != "NaN"})
d = H[H.file.isin([k for k, v in tf.items() if v != "NaN"])]
print("RA-APs by the mixture in those files:", d.groupby("file").ra.sum().to_dict())
print("variant AP rule (peak >= threshold + 30 mV): APs",
      int(sum(r["aps"].variant30.sum() for r in hrows if len(r["aps"]))), "of which also main",
      int(sum((r["aps"].variant30 & r["aps"].main).sum() for r in hrows if len(r["aps"]))))

**Result: partially reproduced, resting on the mixture classes.** 11 cells have AIS-APs and 21 RA-APs, against 7 and 6. With the mean potential before threshold, AIS-APs are all in up states in 10 of 11 cells and RA-APs reach down or transition states in 11 of 21 cells, which meets the criterion's partial rule.

## fig3f-human-pre-ap-fluctuation

> In active human slices, membrane potential fluctuations (SD) 0 to 50 ms before AIS-APs are
> larger than before RA-APs (1.49 against 0.78 mV).

The SD of the membrane potential in the 50 ms before each AP, the window ending at the threshold
or at the peak (the criterion's readings), APs pooled across cells, classes as above. APs with
less than 50 ms of trace before them are left out.

In [ ]:
def sd_compare(D, label):
    """Pooled and per-cell comparison; the variant 'isolated' keeps APs with no AP in the 50 ms
    before their threshold, so that the window holds no spike (not in the paper; shown because a
    preceding spike inflates the SD)."""
    D = D.sort_values(["file", "series", "t_thr"]).copy()
    prev = D.groupby(["file", "series"]).t_peak.shift(1)
    D["isolated"] = (D.t_thr - prev).fillna(np.inf) > 0.05
    for col in ("sd_thr", "sd_peak"):
        a, b = D[~D.ra & D.isolated][col].dropna(), D[D.ra & D.isolated][col].dropna()
        p = stats.mannwhitneyu(a, b).pvalue if len(a) and len(b) else np.nan
        print(f"{label} [isolated APs] window to {col[3:]}: AIS-AP {a.median():.2f} mV (n={len(a)}), "
              f"RA-AP {b.median():.2f} mV (n={len(b)}), Mann-Whitney P = {p:.2g}")
        a, b = D[~D.ra][col].dropna(), D[D.ra][col].dropna()
        p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue if len(a) and len(b) else np.nan
        pc = D.dropna(subset=[col]).groupby(["file", "ra"])[col].median().unstack()
        pcp = stats.mannwhitneyu(pc[False].dropna(), pc[True].dropna()).pvalue if True in pc and pc[True].notna().sum() > 0 else np.nan
        print(f"{label} window to {col[3:]}: AIS-AP SD median {a.median():.2f} mV (n={len(a)}), RA-AP "
              f"{b.median():.2f} mV (n={len(b)}), Mann-Whitney P = {p:.2g}; per-cell medians "
              f"P = {pcp:.2g} ({pc[False].notna().sum()} and {pc[True].notna().sum() if True in pc else 0} cells)")


sd_compare(H, "human")
print("expected: AIS 1.49 mV, RA 0.78 mV (tolerance 10%), Mann-Whitney P < 0.001")

**Result: partially reproduced, resting on the mixture classes.** The SD before AIS-APs is higher (6.42 against 3.78 mV, Mann-Whitney P = 7.3e-09), but both medians are far above 1.49 and 0.78 mV, and APs without a spike in the preceding 50 ms reverse the order (2.16 against 2.47 mV).

## Rat active slices with layer 5 LFP

The condition of each recording is read from `general/pharmacology`: carbachol (also spelled
`Charbachol`) with SCH23390 is the active slice of Figure 3; recordings with neither are the
slices without modulators of Figure S3. Some recordings list further drugs (kainic acid,
gabazine, 4-AP); they are kept in the main set and left out in a variant. APs of the gap-free
sweeps of at least 5 s (`long_sweep_aps`), classed by reading (b), a mixture fitted to these APs
of all rat slice recordings of this folder pooled. The variant keeps only APs without steady
current (at most 5 pA above the holding level at threshold).

In [ ]:
def cond(m):
    drugs = {v.lower().replace("charbachol", "carbachol") for k, v in m["pharmacology"].items()
             if k.startswith("DrugName") and v != "NaN"}
    if {"carbachol", "sch23390"} <= drugs:
        return "modulators" + ("+" + "+".join(sorted(drugs - {"carbachol", "sch23390"})) if drugs - {"carbachol", "sch23390"} else "")
    if not drugs:
        return "none"
    return "+".join(sorted(drugs))


def long_sweep_aps(r, min_s=5.0):
    """APs of the gap-free sweeps (at least 5 s; a standard choice), with or without a steady
    current: the Methods say some cells needed steady somatic current to reach AIS-AP threshold
    ("In vitro oscillations")."""
    a, s = r["aps"], r["sw"]
    if not len(a):
        return a
    long_ = set(s[(s.n / s.rate >= min_s) & s.chan.str.startswith("Vm")].series)
    return a[a.main & a.series.isin(long_)]


R = []
for r in rows_osc:
    r["cond"] = cond(r["meta"])
    a = long_sweep_aps(r)
    if len(a):
        R.append(a.assign(file=r["file"], cond=r["cond"]))
R = pd.concat(R)
R["ra"] = raf.gmm_classes(R, seed("fig3l-rat-spontaneous-ra-proportion"))
print("recordings by condition:", dict(collections.Counter(r["cond"] for r in rows_osc)))
print(f"APs of gap-free sweeps {len(R)}, RA-APs {int(R.ra.sum())}; of these without steady current "
      f"{int((R.i_thr <= 5).sum())}, RA-APs among them {int((R.ra & (R.i_thr <= 5)).sum())}")
print(R.groupby("ra")[["thr", "slope"]].median().round(2))

## fig3i-rat-lfp-rhythm

> In active rat slices, layer 5 LFP shows rhythmic low-frequency deflections at 1.1 Hz,
> synchronized to up states of the layer 1 interneurons.

Per recording, LFP segments of at least 8 s, resampled to 500 Hz. Reading (i): the peak of the
Welch power spectrum (8 s windows, averaged over segments) in 0.5 to 4 Hz; reading (ii): the
number of cycles of the 0.5 to 4 Hz Butterworth-filtered LFP (order 2, zero phase, as in the
header) per second, counted as wraps of the Hilbert phase. Median over recordings.

In [ ]:
FS = 500.0


def lfp_segments(r, min_s=8.0):
    out = []
    for name, (st, rate, x) in sorted(r["lfp"].items(), key=lambda kv: kv[1][0]):
        if len(x) / rate < min_s:
            continue
        q = int(round(rate / FS))
        y = signal.decimate(x.astype(float), q, ftype="fir") if q > 1 else x.astype(float)
        out.append((st, rate / q, y))
    return out


def rhythm(r, band=(0.5, 4.0), order=2):
    segs = lfp_segments(r, 8.0 if band[0] >= 0.5 else 40.0)
    if not segs:
        return dict(n_seg=0, lfp_s=0.0)
    nper = int((8 if band[0] >= 0.5 else 40) * FS)
    ps = []
    cyc, dur = 0.0, 0.0
    sos = signal.butter(order, band, btype="band", fs=FS, output="sos")
    for st, fs, y in segs:
        f, p = signal.welch(y - y.mean(), fs=fs, nperseg=min(nper, len(y)))
        ps.append(np.interp(np.arange(0, 10, 0.025), f, p))
        ph = np.unwrap(np.angle(signal.hilbert(signal.sosfiltfilt(sos, y))))
        cyc += (ph[-1] - ph[0]) / (2 * np.pi)
        dur += len(y) / fs
    f = np.arange(0, 10, 0.025)
    P = np.mean(ps, axis=0)
    m = (f >= band[0]) & (f <= band[1])
    return dict(n_seg=len(segs), lfp_s=dur, peak_hz=f[m][np.argmax(P[m])], cycles_hz=cyc / dur)


rh = pd.DataFrame([dict(file=r["file"][:32], cond=r["cond"], **rhythm(r)) for r in rows_osc])
print(rh.round(3).to_string())

In [ ]:
for label, d in (("carbachol + SCH23390, all", rh[rh.cond.str.startswith("modulators")]),
                 ("carbachol + SCH23390 only", rh[rh.cond == "modulators"])):
    d = d[d.n_seg > 0]
    print(f"{label}: recordings with LFP {len(d)}; (i) spectral peak median {d.peak_hz.median():.2f} Hz "
          f"(SD {d.peak_hz.std():.2f}); (ii) cycle rate median {d.cycles_hz.median():.2f} Hz (SD {d.cycles_hz.std():.2f})")
print("expected: 1.1 Hz (tolerance 0.11)")

**Result: reproduced.** Over 36 recordings the LFP's spectral peak is at a median 1.06 Hz and its cycle rate 1.01 Hz, both within 1.1 +- 0.11 Hz.

## fig3l-rat-spontaneous-ra-proportion

> In active rat slices, 13 of 40 layer 1 interneurons fire spontaneous RA-APs, and all recovered
> ones (n = 5) are NGFCs.

In [ ]:
for label, keep in (("carbachol + SCH23390, all", lambda c: c.startswith("modulators")),
                    ("carbachol + SCH23390 only", lambda c: c == "modulators")):
    files = [r["file"] for r in rows_osc if keep(r["cond"])]
    d = R[R.file.isin(files)]
    per = d.groupby("file").ra.sum().reindex(files, fill_value=0)
    print(f"{label}: cells {len(files)}, with at least one RA-AP {int((per > 0).sum())}, with at least two "
          f"{int((per > 1).sum())}; RA-APs per cell {sorted(per[per > 0].astype(int).tolist())}")
print("expected: 13 cells (tolerance 1.3) of 40. Cell morphology: no cell type field in these files.")

**Result: not reproduced.** The mixture labels RA-APs in 30 of 36 active-slice cells (24 of 30 without extra drugs), against 13 of 40.

## fig3k-rat-pre-ap-fluctuation

> In active rat slices, AIS-APs occur during up states with larger membrane potential
> fluctuations before them, and RA-APs during down states with smaller ones.

In [ ]:
RA_ = R[R.cond.str.startswith("modulators")]
sd_compare(RA_, "rat")
print("The state of each rat AP is not computed here: the rat files' sweeps without current are")
print("the same kind, but the claim does not judge it (criterion).")

**Result: reproduced under one reading, resting on the mixture classes.** With the window ending at the peak the SD before AIS-APs is higher (9.90 against 8.76 mV, P = 1.4e-33); with the window ending at threshold the order reverses (7.70 against 7.97 mV); per-cell medians do not differ (P = 0.22).

## fig3m-rat-ais-phase-coupling and fig3m-rat-ra-no-phase

> In active rat slices, AIS-APs are coupled near the trough of the slow LFP oscillation (mean
> vector length 0.093, angle 314 degrees). RA-APs show no phase preference.

The Methods' phase procedure (header of the claims file): for each AP, the 4 s LFP segment
centred on its peak; FFT power times frequency; the AP is kept when the power in 0.5 to 4 Hz over
that in 0.5 to 45 Hz exceeds 0.1. The LFP segment is band-passed 0.5 to 4 Hz (Butterworth order
2, zero phase; order 4 the variant), Hilbert phase at the AP peak, shifted by 180 degrees so that
the trough is 0 and 360 degrees and the peak 180 (Methods convention). Rayleigh test on the APs
pooled across cells.

In [ ]:
def ap_phases(r, aps, order=2):
    out = []
    for name, (st, rate, x) in r["lfp"].items():
        dur = len(x) / rate
        sel = aps[(aps.t_peak_abs >= st + 2) & (aps.t_peak_abs <= st + dur - 2)]
        if not len(sel):
            continue
        q = int(round(rate / FS))
        y = signal.decimate(x.astype(float), q, ftype="fir") if q > 1 else x.astype(float)
        fs = rate / q
        sos = signal.butter(order, (0.5, 4.0), btype="band", fs=fs, output="sos")
        ph = np.angle(signal.hilbert(signal.sosfiltfilt(sos, y)), deg=True)
        for _, ap in sel.iterrows():
            i = int(round((ap.t_peak_abs - st) * fs))
            h = int(round(2 * fs))
            if i - h < 0 or i + h > len(y):
                continue
            seg = y[i - h:i + h]
            F = np.fft.rfftfreq(len(seg), 1 / fs)
            P = np.abs(np.fft.rfft(seg - seg.mean())) ** 2 * F
            ratio = P[(F >= 0.5) & (F <= 4)].sum() / P[(F >= 0.5) & (F <= 45)].sum()
            out.append(dict(file=r["file"], cond=r["cond"], ra=ap.ra, ratio=ratio, phase=(ph[i] + 180) % 360))
    return pd.DataFrame(out)


R["t_peak_abs"] = R.sweep_start + R.t_peak
PH = {}
for order in (2, 4):
    parts = []
    for r in rows_osc:
        if not r["cond"].startswith("modulators"):
            continue
        d = R[R.file == r["file"]]
        if len(d):
            parts.append(ap_phases(r, d, order))
    PH[order] = pd.concat(parts)
print({o: len(v) for o, v in PH.items()}, "APs with a 4 s LFP segment")

In [ ]:
def phase_report(P, label):
    for ra, name in ((False, "AIS-APs"), (True, "RA-APs")):
        d = P[P.ra == ra]
        k = d[d.ratio > 0.1]
        rr, ang, p, n = raf.rayleigh(k.phase)
        print(f"{label} {name}: with LFP {len(d)} from {d.file.nunique()} cells; kept by delta ratio {n} "
              f"from {k.file.nunique()} cells; vector length {rr:.3f}, angle {ang:.0f} deg, Rayleigh P = {p:.2g}")
        if len(k):
            print("   per cell (kept APs):", k.groupby("file").size().sort_values(ascending=False).head(25).to_dict())


phase_report(PH[2], "order 2")
phase_report(PH[4], "order 4")
phase_report(PH[2][PH[2].cond == "modulators"], "order 2, carbachol + SCH23390 only")
print("expected AIS-APs: length 0.093 (tolerance 0.0093), angle 314 deg (within 31.4), P < 0.001, n = 534 from 22 cells")
print("expected RA-APs: P = 0.07 (no preference at P < 0.05), n = 76 from 22 cells")

**Result: partially reproduced, resting on the choice of population.** Without the recordings with extra drugs, 111 AIS-APs from 7 cells are coupled near the trough (321 degrees, Rayleigh P = 5e-14) with a vector length of 0.508, not 0.093; with all 36 recordings the angle is 245 degrees.

In [ ]:
d = PH[2][PH[2].ra & (PH[2].ratio > 0.1)]
rr, ang, p, n = raf.rayleigh(d.phase)
print(f"RA-APs, order 2: n {n}, vector length {rr:.3f}, angle {ang:.0f} deg, Rayleigh P = {p:.3g}")
d2 = PH[2][PH[2].ra & (PH[2].ratio > 0.1) & (PH[2].cond == "modulators")]
print("order 2, carbachol + SCH23390 only: n %d, length %.3f, angle %.0f, P = %.3g" % (raf.rayleigh(d2.phase)[3], *raf.rayleigh(d2.phase)[:3]))
d4 = PH[4][PH[4].ra & (PH[4].ratio > 0.1)]
print("order 4: n %d, length %.3f, angle %.0f, P = %.3g" % (raf.rayleigh(d4.phase)[3], *raf.rayleigh(d4.phase)[:3]))

**Result: not reproduced.** RA-APs are phase coupled in every population (all recordings: 10311 APs, length 0.203, P = 2.7e-186; without extra drugs: 818 APs, length 0.604, P = 9e-145).

## figs3-no-modulators

> In rat slices in in vivo-like ACSF without carbachol and SCH23390 (n = 90), elevated network
> activity with low-frequency up states (0.11 Hz) appears and layer 1 interneurons fire
> spontaneous RA-APs (4 NGFCs and 5 non-NGFCs).

In [ ]:
nm = [r for r in rows_osc if r["cond"] == "none"]
print(f"recordings without carbachol and SCH23390: {len(nm)} (paper: 90)")
rows = []
for r in nm:
    x = rhythm(r, band=(0.05, 4.0))
    per = R[(R.file == r["file"])].ra.sum()
    rows.append(dict(file=r["file"][:32], ra_aps=int(per), **x))
d = pd.DataFrame(rows)
print(d.round(3).to_string())
if len(d):
    print(f"median spectral peak {d.peak_hz.median():.3f} Hz, cycle rate {d.cycles_hz.median():.3f} Hz; "
          f"cells with RA-APs {int((d.ra_aps > 0).sum())}")
print("expected: 0.11 Hz (tolerance 0.011), 9 cells with RA-APs")
print(f"notebook time {(time.time() - T0) / 60:.1f} min, peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

**Result: partially reproduced.** Only 2 recordings without modulators are in the dandiset (paper 90). Both have RA-APs; the slow rhythm peaks at 0.05 Hz against 0.11 Hz, and 2 cells against 9.

## Summary

| Claim | Outcome |
|---|---|
| fig2c-zd7288-reduces-raf | not reproduced |
| figs2-zd7288-passive-properties | not reproduced |
| fig1hk-bleb-initiation-order | not reproduced |
| fig2h-bleb-sag-higher | partially reproduced |
| fig3-human-ap-states | partially reproduced (assumed) |
| fig3f-human-pre-ap-fluctuation | partially reproduced (assumed) |
| fig3i-rat-lfp-rhythm | reproduced |
| fig3l-rat-spontaneous-ra-proportion | not reproduced (assumed) |
| fig3k-rat-pre-ap-fluctuation | reproduced (assumed) |
| fig3m-rat-ais-phase-coupling | partially reproduced (assumed) |
| fig3m-rat-ra-no-phase | not reproduced (assumed) |
| figs3-no-modulators | partially reproduced |
| in vivo claims (9) | blocked, too_large |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The dandiset has no AP labels, so AIS-APs and RA-APs are separated by a two-component mixture on threshold and pre-AP slope (reading (b) of the claims file), which in these data puts most APs of the rat slices in the RA class and gives the human RA class the steeper slope; every outcome on spontaneous activity rests on it. The ZD7288 comparison splits each recording at the wash-in time the files give. The bleb latency uses the bleb trace around each somatic AP, since bleb APs are too small to detect on their own. The variant with the bleb trace during a somatic step and the seed range of the mixtures were not run.

What was found about the dataset. The voltage series state a conversion of 1e-12 while the values are in volts; many stimulus series are stored in pA under a unit of amperes. Sampling rates range from 5 to 100 kHz in these folders. Cell classes are recorded only for the bleb recordings. One bleb file has no bleb sweeps. Only 2 of the 90 recordings without modulators are present, and several active rat recordings carry drugs the paper does not mention (gabazine, kainic acid, 4-AP).
